# Phase 10 — Color-Invariant Fine-Tune (Colab A100)

Brief fine-tune of the Phase 6 best_model.pth with HSV jitter augmentation.
Pre-registered in `docs/phase10/01_preregistration.md`.

**Expected runtime on Colab A100**: ~90 min total
- Repo clone + deps: 5 min
- Data download (61K images): 30-40 min
- Fine-tune (5 epochs): 60-75 min
- Save ckpt: 1 min

**Requirements**:
- Colab Pro (A100 GPU)
- ~50 GB disk (Colab gives 100 GB)
- ISIC archive public access (no login)

**Steps**:
1. Run all cells in order
2. Download the fine-tuned ckpt (`best_model_color_invariant.pth`) at the end
3. Upload to local repo `artifacts/phase10/` for Phase 9 re-validation

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone repo + checkout Phase 10 branch

In [ ]:
%cd /content
!git clone https://github.com/landerban/melanoma-screening-cnn.git
%cd melanoma-screening-cnn
!git checkout feat/phase9-shortcut-disentanglement
!git pull
!git log --oneline -5

## 3. Install dependencies

In [ ]:
# Colab already has torch + most scientific libs; add the few we need
!pip install -q isic-cli scikit-image opencv-python tqdm
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4. Upload best_model.pth (Phase 6 base ckpt)

Upload `best_model.pth` from your local repo. It's ~18 MB.

**Option A** (interactive): use the file picker below.

**Option B** (if you've put it in Google Drive): mount Drive and copy.

In [ ]:
# Option A — interactive upload
from google.colab import files
print('Upload best_model.pth (18 MB) from your laptop...')
uploaded = files.upload()
import shutil, os
for fn in uploaded:
    if fn != 'best_model.pth':
        shutil.move(fn, 'best_model.pth')
print('Files now in repo root:', [f for f in os.listdir('.') if f.endswith('.pth')])

## 5. Download ISIC training data (3 collections, 60K images, ~50 GB)

This takes ~30-40 min. Run unattended.

In [ ]:
import os
os.makedirs('training_data/images', exist_ok=True)
# Download metadata first (fast)
!python scripts/rebuild_per_collection_metadata.py training_data

In [ ]:
# Download images for all 3 collections
!isic image download -c 212 training_data/images
!isic image download -c 70  training_data/images
!isic image download -c 249 training_data/images
import os
n_imgs = len(os.listdir('training_data/images'))
print(f'\nDownloaded {n_imgs} images')

## 6. Run Phase 10 fine-tune (5 epochs, ~75 min on A100)

In [ ]:
!python scripts/phase10/finetune_color_invariant.py \
    --base-ckpt best_model.pth \
    --out-ckpt artifacts/phase10/best_model_color_invariant.pth \
    --log-path artifacts/phase10/01_finetune.log \
    --epochs 5 \
    --lr 1e-5 \
    --batch-size 96 \
    --num-workers 4

## 7. Inspect the result

In [ ]:
!cat artifacts/phase10/01_finetune.log

In [ ]:
import torch
ckpt = torch.load('artifacts/phase10/best_model_color_invariant.pth',
                  map_location='cpu', weights_only=False)
print('Pre-FT test AUC:', ckpt['pre_finetune_test_auc'])
print('Post-FT test AUC:', ckpt['post_finetune_test_auc'])
print('Best val AUC:', ckpt['best_val_auc'])
print('Best epoch:', ckpt['best_epoch'])
print('Phase 10 mods:', ckpt['phase10_modifications'])

## 8. Download the fine-tuned ckpt back to your laptop

In [ ]:
from google.colab import files
files.download('artifacts/phase10/best_model_color_invariant.pth')
files.download('artifacts/phase10/01_finetune.log')

## 9. (Optional) Save to Google Drive as backup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('artifacts/phase10/best_model_color_invariant.pth',
            '/content/drive/MyDrive/best_model_color_invariant.pth')
shutil.copy('artifacts/phase10/01_finetune.log',
            '/content/drive/MyDrive/phase10_finetune.log')
print('Saved to Drive')